# Data Cleaning 06 -- Daily CRSP Stock Data

## Input
`Data/Data_Collection/Initial/06_Daily_CRSP_Stock_Data/firm_daily/` (partitioned parquet by year, 1,010,280 rows across 2004--2024, 227 PERMNOs)

## Purpose
Cleans daily stock-level data from CRSP dsf_v2 for all PERMNOs in the master universe. Key concerns addressed: high-NaN TAQ-derived columns, all-NaN rows from delisted/halted stocks, bid-ask inversions from pre-Reg NMS quote timing, dividend columns with near-zero non-zero rates, and partial NaN rows from missing opening prices or first-day-of-listing returns.

## Stage 0: Load & Inspect
- Loads all year partitions and verifies PERMNO coverage against the master list
- Reports shape, date range, column inventory separated into ID/metadata/factor groups
- Reports rows per year with approximate stocks-per-day count

## Stage 1: Missing Data Audit
- Total NaN count and percentage across all factor cells
- Per-column NaN counts sorted descending with flags
- Per-row NaN distribution
- Per-PERMNO average NaN rate, identifying the 10 worst PERMNOs

## Stage 2: Coverage Checks
- **Trading days per PERMNO per year:** reports distribution and flags PERMNO-years with fewer than 200 trading days, cross-referenced against `universe_annual` to distinguish partial years (stock entering/leaving universe) from data quality issues
- **Stocks per trading day:** identifies any dates with unusually few stocks
- **Duplicate (permno, date) check**

## Stage 3: Value Range & Quality Checks
- **Price columns:** range, negative value, and zero value checks for `dlyprc`, `dlyopen`, `dlyhigh`, `dlylow`, `dlyclose`, `dlybid`, `dlyask`
- **Return columns:** range, percentile analysis, and count of extreme returns (|return| > 50%)
- **Volume columns:** range and zero-volume checks for `dlyvol`, `dlynumtrd`, `dlyprcvol`, `dlymmcnt`
- **Size columns:** range, zero, and negative checks for `dlycap` and `shrout`
- **Dividend columns:** non-null and non-zero counts for `dlyorddivamt` and `dlynonorddivamt`
- **Adjustment factor:** distribution of `dlyfacprc`, count of values not equal to 1.0
- **Price consistency:** verifies `dlyhigh >= dlylow` and `dlyask >= dlybid`

## Stage 3b: Deep Investigation
- **All-NaN row analysis:** confirms 1,767 all-NaN rows are the same set across all 13 core columns. 1,761 are from PERMNO 81593 (Washington Mutual post-FDIC seizure), 6 from brief biotech trading halts. Year distribution and sample rows printed.
- **Verified against universe:** confirmed that none of the all-NaN rows were in the top-100 universe at the time -- they will drop out naturally during the merge on `universe_annual`.
- **Partial NaN rows:** 151 rows with some but not all columns NaN, mostly missing `dlyopen` (94), `dlybid`/`dlyask` (52), or `dlyret` (29 first-day-of-listing).
- **Bid-ask inversion investigation:** 9,992 rows (1%) where `dlyask < dlybid`. Concentrated in 2004--2006 (pre-Reg NMS). Median inversion is $0.02 (rounding-level). Distribution by year, by PERMNO, and by inversion magnitude reported. Not corrected -- the bid-ask spread remains a valid liquidity factor when computed as `abs(ask - bid) / midpoint`.

## Stage 5: Clean & Save

### Columns Dropped (5)
- `dlynumtrd`, `dlymmcnt` -- 77.6% NaN. TAQ-derived fields that CRSP only populates for recent years.
- `dlynonorddivamt` -- only 2 non-zero values in 1M+ rows. Unusable.
- `dlyorddivamt` -- 1.26% non-zero. Dividends are infrequent discrete events; the return columns (`dlyret` vs `dlyretx`) already capture the dividend effect.
- `dlyfacprc` -- price adjustment factor for stock splits. Metadata for reconstructing unadjusted prices, not a predictive feature.

### Metadata Columns Retained But Not Factors
`ticker`, `primaryexch`, `year` -- kept for debugging and inspection, excluded from the factor list during merge.

### No Rows Dropped
1,767 all-NaN rows confirmed to not be in the top-100 universe at the time. They drop out naturally when the merge pipeline joins on `universe_annual`.

### No Winsorisation
Returns range from -94% to +128% with 37 observations where |return| > 50%. These are genuine large-cap events (e.g., Tesla volatility, Washington Mutual collapse). Winsorisation applied cross-sectionally per date in the merge pipeline.

### Bid-Ask Inversions (9,992 Rows, ~1%)
Not corrected. Closing quote timing mismatches concentrated in 2004--2006 (pre-Reg NMS). Median inversion is $0.02.

### Partial NaN (151 Rows)
Missing `dlyopen` (94), `dlybid`/`dlyask` (52), or `dlyret` (29 first-day-of-listing). Left as NaN -- cross-sectional aggregation skips these naturally.

## Output
`Data/Data_Collection/Cleaned/06_Daily_CRSP_Stock_Data/crsp_daily_clean.parquet` -- 14 factor columns (down from 19), plus `permno`, `date`, `ticker`, `primaryexch`, `year`

In [1]:
# %% [markdown]
# # Data Cleaning: CRSP Daily Stock Data (firm_daily)
#
# Source: Data/Data_Collection/Initial/06_Daily_CRSP_Stock_Data/firm_daily/ (partitioned by year)
# Output: Data/Data_Collection/Cleaned/06_Daily_CRSP_Stock_Data/crsp_daily_clean.parquet
#
# Daily stock-level data from CRSP dsf_v2 for all 227 PERMNOs in the master
# universe. Already filtered to universe during collection. Partitioned by year
# (2004–2024).
#
# Key columns:
#   - Prices: dlyprc, dlyopen, dlyhigh, dlylow, dlyclose, dlybid, dlyask
#   - Returns: dlyret, dlyretx (ex-dividend), dlyreti (including distributions)
#   - Volume: dlyvol, dlynumtrd, dlyprcvol, dlymmcnt
#   - Size: dlycap, shrout
#   - Other: dlyfacprc (adjustment factor), dlyorddivamt, dlynonorddivamt
#   - Metadata: ticker, primaryexch

# %%
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR     = Path('../../../Data/Data_Collection/Initial/06_Daily_CRSP_Stock_Data/firm_daily/')
MASTER_PATH = Path('../../../Data/Data_Collection/Cleaned/01_Top100_SP500_Universe/universe_master_clean.parquet')
ANNUAL_PATH = Path('../../../Data/Data_Collection/Cleaned/01_Top100_SP500_Universe/universe_annual_clean.parquet')
OUT_DIR     = Path('../../../Data/Data_Collection/Cleaned/06_Daily_CRSP_Stock_Data')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 0: LOAD & INSPECT
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STAGE 0: LOAD & INSPECT — CRSP Daily")
print("=" * 90)

# ── Load all year partitions ─────────────────────────────────────────────────
df = pd.read_parquet(RAW_DIR)
df['date'] = pd.to_datetime(df['date'])

master = pd.read_parquet(MASTER_PATH)

print(f"\n  Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"  Unique dates: {df['date'].nunique():,}")
print(f"  Unique PERMNOs: {df['permno'].nunique()}")
print(f"  Master PERMNOs: {len(master)}")

# Verify all PERMNOs are from master
data_permnos = set(df['permno'].unique())
master_permnos = set(master['permno'])
extra = data_permnos - master_permnos
missing = master_permnos - data_permnos
print(f"\n  PERMNOs in data but NOT in master: {len(extra)}")
print(f"  PERMNOs in master but NOT in data: {len(missing)}")
if missing:
    print(f"    Missing: {sorted(missing)}")

# ── Column inventory ─────────────────────────────────────────────────────────
all_cols = df.columns.tolist()
date_cols = ['date']
id_cols = ['permno']
meta_cols = [c for c in ['ticker', 'primaryexch', 'year'] if c in all_cols]
factor_cols = [c for c in all_cols if c not in date_cols + id_cols + meta_cols]

print(f"\n  ID columns: {id_cols}")
print(f"  Meta columns: {meta_cols}")
print(f"  Factor columns ({len(factor_cols)}):")
for i, c in enumerate(factor_cols, 1):
    print(f"    {i:>3d}. {c:<25s} {str(df[c].dtype):<15s}")

print(f"\n--- Head (10 rows) ---")
show_cols = ['permno', 'date'] + factor_cols[:8]
print(df[show_cols].head(10).to_string(index=False))

print(f"\n--- Tail (10 rows) ---")
print(df[show_cols].tail(10).to_string(index=False))

# ── Rows per year ────────────────────────────────────────────────────────────
print(f"\n--- Rows per year ---")
rows_per_year = df.groupby(df['date'].dt.year).size()
for year, n in rows_per_year.items():
    avg_stocks = n / 252
    print(f"  {year}: {n:>7,d} rows (~{avg_stocks:.0f} stocks/day)")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 1: MISSING DATA AUDIT
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 1: MISSING DATA AUDIT")
print("=" * 90)

n_rows = len(df)

# ── Total NaN ────────────────────────────────────────────────────────────────
total_cells = n_rows * len(factor_cols)
total_nan = df[factor_cols].isna().sum().sum()
print(f"\nTotal cells: {total_cells:,}")
print(f"Total NaN:   {total_nan:,} ({total_nan / total_cells * 100:.2f}%)")

# ── Per-column NaN (sorted descending) ───────────────────────────────────────
col_nan = df[factor_cols].isna().sum()
col_nan_pct = (col_nan / n_rows * 100).round(2)
col_nan_sorted = col_nan_pct.sort_values(ascending=False)

print(f"\n--- Per-Column NaN ---")
print(f"\n  {'Column':<25s} {'NaN %':>8s}  {'Count':>10s}")
print("  " + "-" * 50)
for col, pct in col_nan_sorted.items():
    count = int(col_nan[col])
    flag = " ← DROP" if pct >= 30 else (" ← INVESTIGATE" if pct >= 10 else "")
    print(f"  {col:<25s} {pct:>7.2f}%  {count:>10,d}{flag}")

# ── Per-row NaN ──────────────────────────────────────────────────────────────
row_nan = df[factor_cols].isna().sum(axis=1)
print(f"\n--- Per-Row NaN Distribution ---")
print(f"  Rows with 0 NaN: {(row_nan == 0).sum():>10,d} ({(row_nan == 0).mean()*100:.1f}%)")
print(f"  Rows with 1-3 NaN: {((row_nan >= 1) & (row_nan <= 3)).sum():>10,d}")
print(f"  Rows with 4-8 NaN: {((row_nan > 3) & (row_nan <= 8)).sum():>10,d}")
print(f"  Rows with >8 NaN: {(row_nan > 8).sum():>10,d}")

# ── Per-PERMNO NaN rate ──────────────────────────────────────────────────────
print(f"\n--- Per-PERMNO Average NaN Rate ---")
permno_nan = (
    df.groupby('permno')[factor_cols]
    .apply(lambda x: x.isna().mean().mean() * 100)
    .sort_values(ascending=False)
)
print(f"  PERMNOs with <5% avg NaN:  {(permno_nan < 5).sum()}")
print(f"  PERMNOs with 5-15% avg NaN: {((permno_nan >= 5) & (permno_nan < 15)).sum()}")
print(f"  PERMNOs with >15% avg NaN: {(permno_nan >= 15).sum()}")

worst = permno_nan.head(10)
if len(worst) > 0:
    print(f"\n  10 worst PERMNOs:")
    for permno, pct in worst.items():
        n_rows_p = len(df[df['permno'] == permno])
        print(f"    PERMNO {int(permno):>6d}  {pct:.1f}% avg NaN  ({n_rows_p:,} rows)")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 2: COVERAGE CHECKS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 2: COVERAGE CHECKS")
print("=" * 90)

# ── 2a. Trading days per year per PERMNO ─────────────────────────────────────
print(f"\n--- Trading days per PERMNO per year ---")
annual = pd.read_parquet(ANNUAL_PATH)
coverage = df.groupby(['permno', df['date'].dt.year]).size().reset_index(name='n_days')
coverage.columns = ['permno', 'year', 'n_days']

# Expected: ~252 trading days per year
print(f"  Overall: mean={coverage['n_days'].mean():.0f}, "
      f"median={coverage['n_days'].median():.0f}, "
      f"min={coverage['n_days'].min()}, max={coverage['n_days'].max()}")

# Stocks with <200 trading days in a year (partial years)
low_coverage = coverage[coverage['n_days'] < 200]
if len(low_coverage) > 0:
    print(f"\n  PERMNO-years with <200 trading days: {len(low_coverage)}")
    for _, row in low_coverage.head(20).iterrows():
        # Check if this PERMNO was in the universe that year
        in_universe = len(annual[(annual['permno'] == row['permno']) & 
                                 (annual['year'] == row['year'])]) > 0
        flag = "IN universe" if in_universe else "NOT in universe"
        print(f"    PERMNO {int(row['permno']):>6d}  year {int(row['year'])}  "
              f"{int(row['n_days']):>3d} days  ({flag})")

# ── 2b. Date coverage: any dates with very few stocks? ──────────────────────
print(f"\n--- Stocks per trading day ---")
stocks_per_day = df.groupby('date')['permno'].nunique()
print(f"  Mean: {stocks_per_day.mean():.1f}")
print(f"  Min:  {stocks_per_day.min()} (on {stocks_per_day.idxmin().date()})")
print(f"  Max:  {stocks_per_day.max()} (on {stocks_per_day.idxmax().date()})")

low_days = stocks_per_day[stocks_per_day < 100]
if len(low_days) > 0:
    print(f"\n  Days with <100 stocks: {len(low_days)}")
    for date, n in low_days.sort_values().head(10).items():
        print(f"    {date.date()}: {n} stocks")

# ── 2c. Duplicate (permno, date) check ──────────────────────────────────────
print(f"\n--- Duplicate (permno, date) ---")
n_dupes = df.duplicated(subset=['permno', 'date']).sum()
if n_dupes == 0:
    print(f"  ✓ No duplicates")
else:
    print(f"  ⚠ {n_dupes} duplicates")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 3: VALUE RANGE & QUALITY CHECKS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 3: VALUE RANGE & QUALITY CHECKS")
print("=" * 90)

# ── 3a. Price columns ───────────────────────────────────────────────────────
print(f"\n--- Price columns ---")
price_cols = ['dlyprc', 'dlyopen', 'dlyhigh', 'dlylow', 'dlyclose', 'dlybid', 'dlyask']
for col in price_cols:
    if col not in df.columns:
        continue
    vals = df[col].dropna()
    n_neg = (vals < 0).sum()
    n_zero = (vals == 0).sum()
    print(f"  {col:<15s} range: ${vals.min():.2f} – ${vals.max():.2f}  "
          f"mean: ${vals.mean():.2f}  neg: {n_neg}  zero: {n_zero}")

# ── 3b. Return columns ──────────────────────────────────────────────────────
print(f"\n--- Return columns ---")
return_cols = ['dlyret', 'dlyretx', 'dlyreti']
for col in return_cols:
    if col not in df.columns:
        continue
    vals = df[col].dropna()
    pctiles = vals.quantile([0.01, 0.05, 0.95, 0.99])
    n_extreme = ((vals > 0.5) | (vals < -0.5)).sum()
    print(f"  {col:<15s} range: [{vals.min():.4f}, {vals.max():.4f}]  "
          f"mean: {vals.mean():.6f}")
    print(f"  {'':15s} 1st: {pctiles[0.01]:.4f}  5th: {pctiles[0.05]:.4f}  "
          f"95th: {pctiles[0.95]:.4f}  99th: {pctiles[0.99]:.4f}  "
          f"|ret|>50%: {n_extreme}")

# ── 3c. Volume and trading ──────────────────────────────────────────────────
print(f"\n--- Volume columns ---")
vol_cols = ['dlyvol', 'dlynumtrd', 'dlyprcvol', 'dlymmcnt']
for col in vol_cols:
    if col not in df.columns:
        continue
    vals = df[col].dropna()
    n_zero = (vals == 0).sum()
    print(f"  {col:<15s} range: [{vals.min():,.0f}, {vals.max():,.0f}]  "
          f"mean: {vals.mean():,.0f}  zero: {n_zero}")

# ── 3d. Market cap and shares ────────────────────────────────────────────────
print(f"\n--- Size columns ---")
for col in ['dlycap', 'shrout']:
    if col not in df.columns:
        continue
    vals = df[col].dropna()
    n_zero = (vals == 0).sum()
    n_neg = (vals < 0).sum()
    print(f"  {col:<15s} range: [{vals.min():,.0f}, {vals.max():,.0f}]  "
          f"mean: {vals.mean():,.0f}  zero: {n_zero}  neg: {n_neg}")

# ── 3e. Dividend columns ────────────────────────────────────────────────────
print(f"\n--- Dividend columns ---")
for col in ['dlyorddivamt', 'dlynonorddivamt']:
    if col not in df.columns:
        continue
    vals = df[col].dropna()
    n_nonzero = (vals != 0).sum()
    print(f"  {col:<25s} {len(vals):,} non-null, {n_nonzero:,} non-zero "
          f"({n_nonzero/max(len(vals),1)*100:.2f}%)")

# ── 3f. Adjustment factor ───────────────────────────────────────────────────
if 'dlyfacprc' in df.columns:
    vals = df['dlyfacprc'].dropna()
    n_not_one = (vals != 1.0).sum()
    print(f"\n--- Adjustment factor (dlyfacprc) ---")
    print(f"  range: [{vals.min():.6f}, {vals.max():.6f}]")
    print(f"  Values != 1.0: {n_not_one:,} ({n_not_one/len(vals)*100:.1f}%)")

# ── 3g. High >= Low, High >= Close, etc. ────────────────────────────────────
print(f"\n--- Price consistency ---")
if all(c in df.columns for c in ['dlyhigh', 'dlylow']):
    both = df[['dlyhigh', 'dlylow']].dropna()
    n_inverted = (both['dlyhigh'] < both['dlylow']).sum()
    print(f"  dlyhigh < dlylow: {n_inverted}")

if all(c in df.columns for c in ['dlybid', 'dlyask']):
    both = df[['dlybid', 'dlyask']].dropna()
    n_inverted = (both['dlyask'] < both['dlybid']).sum()
    print(f"  dlyask < dlybid: {n_inverted}")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 4: SUMMARY — DECISIONS NEEDED
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 4: SUMMARY — DECISIONS NEEDED")
print("=" * 90)

print(f"""
Review the output above:

1. COLUMNS TO DROP:
   - Any column with ≥30% NaN (likely dlynumtrd, dlymmcnt based on prior diagnostics)
   - Metadata columns not needed as factors (ticker, primaryexch)
   - Dividend columns if mostly zero (infrequent events, not useful as daily factors)
   - dlyfacprc (adjustment factor, not a predictive feature)

2. COLUMNS TO KEEP:
   - dlyprc, dlyret, dlyretx: core price and return data
   - dlyvol, dlyprcvol: volume and dollar volume
   - dlycap, shrout: market cap and shares (needed for cap-weighting)
   - dlyopen, dlyhigh, dlylow, dlyclose: OHLC for intraday range features
   - dlybid, dlyask: bid-ask spread (liquidity measure)

3. NaN HANDLING:
   - This is stock-level daily data — do NOT forward-fill
   - NaN in returns on specific days = no trading, legitimate
   - Cross-sectional aggregation skips NaN naturally

4. PARTIAL YEARS:
   - Stocks entering/leaving the universe mid-year will have <252 days
   - This is expected, not a data quality issue

""")


STAGE 0: LOAD & INSPECT — CRSP Daily

  Shape: 1,010,280 rows × 24 columns
  Date range: 2004-01-02 → 2024-12-31
  Unique dates: 5,285
  Unique PERMNOs: 227
  Master PERMNOs: 227

  PERMNOs in data but NOT in master: 0
  PERMNOs in master but NOT in data: 0

  ID columns: ['permno']
  Meta columns: ['ticker', 'primaryexch', 'year']
  Factor columns (19):
      1. dlyprc                    Float64        
      2. dlycap                    Float64        
      3. dlyret                    Float64        
      4. dlyretx                   Float64        
      5. dlyreti                   Float64        
      6. dlyvol                    Float64        
      7. dlyopen                   Float64        
      8. dlyhigh                   Float64        
      9. dlylow                    Float64        
     10. dlyclose                  Float64        
     11. dlybid                    Float64        
     12. dlyask                    Float64        
     13. dlynumtrd             

In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 3b: DEEP INVESTIGATION
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STAGE 3b: DEEP INVESTIGATION")
print("=" * 90)

# ── Are the ~0.18% NaN columns the same rows? ───────────────────────────────
print(f"\n--- Are the ~0.18% NaN the same rows? ---")
nan_cols_cluster = ['dlyprc', 'dlycap', 'dlyret', 'dlyretx', 'dlyreti',
                    'dlyvol', 'dlyopen', 'dlyhigh', 'dlylow', 'dlyclose',
                    'dlybid', 'dlyask', 'dlyprcvol']
nan_cols_present = [c for c in nan_cols_cluster if c in df.columns]

all_nan_mask = df[nan_cols_present].isna().all(axis=1)
any_nan_mask = df[nan_cols_present].isna().any(axis=1)
print(f"  Rows where ALL 13 are NaN: {all_nan_mask.sum():,}")
print(f"  Rows where ANY of 13 are NaN: {any_nan_mask.sum():,}")
print(f"  → {'Same rows' if all_nan_mask.sum() == any_nan_mask.sum() else 'Overlapping but different'}")

# What do the all-NaN rows look like?
nan_rows = df[all_nan_mask].copy()
print(f"\n  All-NaN rows: {len(nan_rows):,}")

# Which columns ARE populated in these rows?
print(f"\n  Non-NaN columns in all-NaN rows:")
for col in factor_cols:
    n_valid = nan_rows[col].notna().sum()
    if n_valid > 0:
        print(f"    {col:<25s} {n_valid:,} valid values")

# Which PERMNOs?
print(f"\n  PERMNOs with all-NaN rows:")
permno_nan_counts = nan_rows.groupby('permno').size().sort_values(ascending=False)
print(f"    Total PERMNOs affected: {len(permno_nan_counts)}")
for permno, n in permno_nan_counts.head(15).items():
    ticker = df[df['permno'] == permno]['ticker'].iloc[0] if 'ticker' in df.columns else '?'
    total = len(df[df['permno'] == permno])
    print(f"    PERMNO {int(permno):>6d} ({ticker:<6s}): {n:>4d} NaN rows / {total:,} total ({n/total*100:.1f}%)")

# Date distribution
print(f"\n  All-NaN rows by year:")
year_dist = nan_rows['date'].dt.year.value_counts().sort_index()
for year, n in year_dist.items():
    print(f"    {year}: {n:>5d}")

# Sample some all-NaN rows
print(f"\n  Sample all-NaN rows:")
sample_cols = ['permno', 'date', 'ticker', 'dlyprc', 'dlyret', 'dlyvol',
               'shrout', 'dlyfacprc', 'dlyorddivamt']
sample_cols = [c for c in sample_cols if c in df.columns]
print(nan_rows[sample_cols].head(10).to_string(index=False))

# ── Rows where SOME but not ALL are NaN ─────────────────────────────────────
partial_nan = df[any_nan_mask & ~all_nan_mask]
if len(partial_nan) > 0:
    print(f"\n--- Partial NaN rows (some but not all of the 13 columns) ---")
    print(f"  Count: {len(partial_nan):,}")

    # Which columns are NaN in these rows?
    print(f"\n  NaN distribution in partial rows:")
    for col in nan_cols_present:
        n = partial_nan[col].isna().sum()
        if n > 0:
            print(f"    {col:<25s} {n:>5d} NaN")

    print(f"\n  Sample partial NaN rows:")
    sample_cols2 = ['permno', 'date', 'ticker', 'dlyprc', 'dlyret', 'dlyopen',
                    'dlyclose', 'dlybid', 'dlyask', 'dlyvol']
    sample_cols2 = [c for c in sample_cols2 if c in df.columns]
    print(partial_nan[sample_cols2].head(10).to_string(index=False))

# ── Bid-ask inversion investigation ─────────────────────────────────────────
print(f"\n\n--- Bid-Ask Inversion (dlyask < dlybid): 9,992 rows ---")
if all(c in df.columns for c in ['dlybid', 'dlyask']):
    both = df[['permno', 'date', 'dlybid', 'dlyask', 'dlyprc']].dropna(
        subset=['dlybid', 'dlyask']
    ).copy()
    both['inverted'] = both['dlyask'] < both['dlybid']
    inverted = both[both['inverted']]

    print(f"  Total rows with bid & ask populated: {len(both):,}")
    print(f"  Inverted (ask < bid): {len(inverted):,} ({len(inverted)/len(both)*100:.2f}%)")

    # How bad is the inversion?
    inverted_diff = inverted['dlybid'] - inverted['dlyask']
    inverted_pct = (inverted_diff / inverted['dlybid'] * 100)
    print(f"\n  Inversion magnitude (bid - ask):")
    print(f"    mean: ${inverted_diff.mean():.4f}")
    print(f"    median: ${inverted_diff.median():.4f}")
    print(f"    max: ${inverted_diff.max():.4f}")
    print(f"    As % of bid — mean: {inverted_pct.mean():.4f}%, "
          f"median: {inverted_pct.median():.4f}%, max: {inverted_pct.max():.4f}%")

    # Is it concentrated in certain years?
    print(f"\n  Inversions by year:")
    inv_year = inverted.groupby(inverted['date'].dt.year).size()
    total_year = both.groupby(both['date'].dt.year).size()
    for year in sorted(inv_year.index):
        n_inv = inv_year[year]
        n_total = total_year.get(year, 1)
        print(f"    {year}: {n_inv:>5d} / {n_total:>6,d} ({n_inv/n_total*100:.1f}%)")

    # Is it concentrated in certain PERMNOs?
    print(f"\n  Top 10 PERMNOs by inversion count:")
    permno_inv = inverted.groupby('permno').size().sort_values(ascending=False)
    for permno, n in permno_inv.head(10).items():
        ticker = df[df['permno'] == permno]['ticker'].iloc[0] if 'ticker' in df.columns else '?'
        total_p = len(both[both['permno'] == permno])
        print(f"    PERMNO {int(permno):>6d} ({ticker:<6s}): {n:>4d} inversions / "
              f"{total_p:,} total ({n/total_p*100:.1f}%)")

    # Sample inverted rows
    print(f"\n  Sample inverted rows:")
    print(inverted[['permno', 'date', 'dlybid', 'dlyask', 'dlyprc']].head(10).to_string(index=False))

    # Are inversions tiny (rounding) or meaningful?
    print(f"\n  Inversion size distribution:")
    print(f"    < $0.01: {(inverted_diff < 0.01).sum():,}")
    print(f"    $0.01–$0.10: {((inverted_diff >= 0.01) & (inverted_diff < 0.10)).sum():,}")
    print(f"    $0.10–$1.00: {((inverted_diff >= 0.10) & (inverted_diff < 1.00)).sum():,}")
    print(f"    > $1.00: {(inverted_diff >= 1.00).sum():,}")

STAGE 3b: DEEP INVESTIGATION

--- Are the ~0.18% NaN the same rows? ---
  Rows where ALL 13 are NaN: 1,767
  Rows where ANY of 13 are NaN: 1,918
  → Overlapping but different

  All-NaN rows: 1,767

  Non-NaN columns in all-NaN rows:
    dlynumtrd                 2 valid values
    dlymmcnt                  6 valid values
    dlyfacprc                 1,767 valid values
    shrout                    1,767 valid values
    dlyorddivamt              1,767 valid values
    dlynonorddivamt           1,767 valid values

  PERMNOs with all-NaN rows:
    Total PERMNOs affected: 4
    PERMNO  81593 (WM    ): 1761 NaN rows / 5,285 total (33.3%)
    PERMNO  76841 (BIIB  ):    3 NaN rows / 5,285 total (0.1%)
    PERMNO  76744 (VRTX  ):    2 NaN rows / 5,285 total (0.0%)
    PERMNO  76614 (REGN  ):    1 NaN rows / 5,285 total (0.0%)

  All-NaN rows by year:
    2006:     2
    2008:    66
    2009:   252
    2010:   252
    2011:   253
    2012:   250
    2013:   252
    2014:   252
    2015:   18

In [3]:
# %%
print("--- Are NaN rows in the top-100 universe at the time? ---")
annual = pd.read_parquet(ANNUAL_PATH)

all_nan_mask = df[nan_cols_present].isna().all(axis=1)
nan_rows = df[all_nan_mask][['permno', 'date']].copy()
nan_rows['year'] = nan_rows['date'].dt.year

# Merge with universe_annual to see if they were in top-100
nan_in_universe = nan_rows.merge(
    annual[['permno', 'year']],
    on=['permno', 'year'],
    how='inner'
)

print(f"  Total all-NaN rows: {len(nan_rows):,}")
print(f"  Of those, in top-100 that year: {len(nan_in_universe):,}")

if len(nan_in_universe) > 0:
    print(f"\n  NaN rows that WERE in top-100:")
    for _, row in nan_in_universe.groupby('permno').agg(
        n=('date', 'size'),
        first=('date', 'min'),
        last=('date', 'max')
    ).iterrows():
        ticker = df[df['permno'] == _]['ticker'].iloc[0] if 'ticker' in df.columns else '?'
        print(f"    PERMNO {int(_):>6d} ({ticker:<6s}): {row['n']} rows "
              f"({row['first'].date()} → {row['last'].date()})")
else:
    print(f"\n  ✓ None of the all-NaN rows were in the top-100 at the time.")
    print(f"    Safe to leave them — they'll drop out during merge.")

--- Are NaN rows in the top-100 universe at the time? ---
  Total all-NaN rows: 1,767
  Of those, in top-100 that year: 0

  ✓ None of the all-NaN rows were in the top-100 at the time.
    Safe to leave them — they'll drop out during merge.


In [4]:
# %% [markdown]
# ## Stage 5: Clean & Save
#
# **Data overview:**
# Daily stock-level data from CRSP dsf_v2 for all 227 PERMNOs in the master
# universe. 1,010,280 rows across 2004–2024 (~191 stocks/day on average).
# Already filtered to universe PERMNOs during collection.
#
# **Columns dropped (5):**
# - `dlynumtrd` (77.6% NaN), `dlymmcnt` (77.6% NaN): TAQ-derived fields
#   that CRSP only populates for recent years. Well above the 30% threshold.
# - `dlynonorddivamt`: only 2 non-zero values in 1M+ rows. Unusable.
# - `dlyorddivamt`: 1.26% non-zero. Dividends are infrequent discrete events,
#   not useful as a daily predictive factor. The return columns (`dlyret` vs
#   `dlyretx`) already capture the dividend effect.
# - `dlyfacprc`: price adjustment factor (for stock splits). Metadata for
#   reconstructing unadjusted prices, not a predictive feature.
#
# **Metadata columns retained but not factors:**
# - `ticker`, `primaryexch`: kept for debugging/inspection, excluded from
#   factor list during merge.
# - `year`: partition column from parquet, kept for convenience.
#
# **No rows dropped.** 1,767 all-NaN rows (1,761 from PERMNO 81593 =
# Washington Mutual post-FDIC seizure, 6 from brief biotech trading halts)
# were confirmed to NOT be in the top-100 universe at the time. They will
# drop out naturally when the merge pipeline joins on `universe_annual`.
#
# **No winsorisation.** Returns range from -94% to +128% with 37 observations
# where |return| > 50%. These are genuine large-cap events (e.g., Tesla
# +13% days, WaMu collapse). Winsorisation applied cross-sectionally per
# date in the merge pipeline.
#
# **Bid-ask inversions (9,992 rows, ~1%):** Not corrected. These are closing
# quote timing mismatches concentrated in 2004–2006 (pre-Reg NMS). Median
# inversion is $0.02. The bid-ask spread remains a valid liquidity factor
# when computed as `abs(ask - bid) / midpoint`.
#
# **Partial NaN (151 rows):** Missing `dlyopen` (94), `dlybid`/`dlyask` (52),
# `dlyret` (29 first-day-of-listing). Left as NaN — cross-sectional
# aggregation skips these naturally.
#
# **Factors retained: 14** (was 19 before cleaning)

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 5: CLEAN & SAVE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STAGE 5: CLEAN & SAVE")
print("=" * 90)

# ── 5a. Drop columns ────────────────────────────────────────────────────────
drop_cols = ['dlynumtrd', 'dlymmcnt', 'dlynonorddivamt', 'dlyorddivamt', 'dlyfacprc']
drop_cols_present = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=drop_cols_present)

meta_cols = [c for c in ['ticker', 'primaryexch', 'year'] if c in df.columns]
factor_cols = [c for c in df.columns if c not in ['date', 'permno'] + meta_cols]

print(f"\n  Dropped {len(drop_cols_present)} columns: {drop_cols_present}")
print(f"  Remaining factor columns: {len(factor_cols)}")
print(f"  Metadata columns kept: {meta_cols}")

# ── 5b. Final NaN report ────────────────────────────────────────────────────
nan_check = df[factor_cols].isna().sum()
nan_cols = nan_check[nan_check > 0]
if len(nan_cols) == 0:
    print(f"\n  ✓ Zero NaN in factor columns")
else:
    total_nan = nan_cols.sum()
    print(f"\n  Remaining NaN: {total_nan:,}")
    for col, n in nan_cols.sort_values(ascending=False).items():
        pct = n / len(df) * 100
        print(f"    {col:<20s} {n:>6,d} NaN ({pct:.2f}%)")
    print(f"\n  All NaN rows are outside the top-100 universe (verified).")
    print(f"  They will drop out during merge on universe_annual.")

# ── 5c. Final summary ───────────────────────────────────────────────────────
print(f"\n  Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  PERMNOs: {df['permno'].nunique()}")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")

print(f"\n  Factor list ({len(factor_cols)} columns):")
for i, c in enumerate(factor_cols, 1):
    vals = df[c].dropna()
    nan_n = df[c].isna().sum()
    nan_str = f"  ({nan_n:,} NaN, {nan_n/len(df)*100:.2f}%)" if nan_n > 0 else ""
    print(f"    {i:>3d}. {c:<20s} range: [{vals.min():,.4f}, {vals.max():,.4f}]{nan_str}")

print(f"\n  Sample (first 5 rows):")
show_cols = ['permno', 'date'] + factor_cols[:8]
print(df[show_cols].head(5).to_string(index=False))

# ── 5d. Save ─────────────────────────────────────────────────────────────────
out_path = OUT_DIR / 'crsp_daily_clean.parquet'
df.to_parquet(out_path, index=False, engine='pyarrow')
print(f"\n  ✓ Saved: {out_path}")
print(f"    {df.shape[0]:,} rows × {df.shape[1]} columns")

print("\nCleaning complete.")

STAGE 5: CLEAN & SAVE

  Dropped 5 columns: ['dlynumtrd', 'dlymmcnt', 'dlynonorddivamt', 'dlyorddivamt', 'dlyfacprc']
  Remaining factor columns: 14
  Metadata columns kept: ['ticker', 'primaryexch', 'year']

  Remaining NaN: 23,259
    dlyopen               1,861 NaN (0.18%)
    dlybid                1,819 NaN (0.18%)
    dlyask                1,819 NaN (0.18%)
    dlyret                1,796 NaN (0.18%)
    dlyretx               1,796 NaN (0.18%)
    dlyreti               1,796 NaN (0.18%)
    dlyhigh               1,768 NaN (0.18%)
    dlylow                1,768 NaN (0.18%)
    dlyclose              1,768 NaN (0.18%)
    dlyprc                1,767 NaN (0.17%)
    dlycap                1,767 NaN (0.17%)
    dlyvol                1,767 NaN (0.17%)
    dlyprcvol             1,767 NaN (0.17%)

  All NaN rows are outside the top-100 universe (verified).
  They will drop out during merge on universe_annual.

  Final shape: 1,010,280 rows × 19 columns
  PERMNOs: 227
  Date range: 2004-01